# TACO Dataset Exploration
Run from the **repo root** (not from the `ml/` directory).
All paths are relative to the repo root.

In [ ]:
import json
import os
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

# ── Run from repo root ──────────────────────────────────────────────────
REPO_ROOT = Path().resolve()
while not (REPO_ROOT / 'ml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print(f'Working directory: {REPO_ROOT}')

ANN_PATH  = Path('data/taco/annotations.json')
TACO_DIR  = Path('data/taco')

if not ANN_PATH.exists():
    raise FileNotFoundError(
        f'{ANN_PATH} not found.\n'
        'Download TACO first: see ml/README.md for instructions.'
    )

with ANN_PATH.open() as f:
    coco = json.load(f)

images      = coco['images']
annotations = coco['annotations']
categories  = coco['categories']

cat_id_to_name = {c['id']: c['name'] for c in categories}
print(f'Total images      : {len(images)}')
print(f'Total annotations : {len(annotations)}')
print(f'Total categories  : {len(categories)}')

## 1 — Class Distribution

In [ ]:
class_counts = Counter(cat_id_to_name[a['category_id']] for a in annotations)
sorted_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)
names, counts = zip(*sorted_classes)

fig, ax = plt.subplots(figsize=(18, 6))
bars = ax.bar(range(len(names)), counts, color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=75, ha='right', fontsize=8)
ax.set_ylabel('Number of annotations')
ax.set_title('TACO Class Distribution (all 60 categories)')
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print('\n── Top 5 most common classes ────────────────────────')
for name, count in sorted_classes[:5]:
    print(f'  {name:<35} {count:>5} annotations')

print('\n── Top 5 rarest classes ─────────────────────────────')
for name, count in sorted_classes[-5:]:
    print(f'  {name:<35} {count:>5} annotations')

## 2 — Visualise Sample Images with Bounding Boxes

In [ ]:
# Build image_id → list[annotation] lookup
img_to_anns = {}
for ann in annotations:
    img_to_anns.setdefault(ann['image_id'], []).append(ann)

img_meta = {img['id']: img for img in images}

# Pick 9 random images that exist on disk
random.seed(42)
candidates = [img for img in images if img_to_anns.get(img['id'])]
sample_imgs = random.sample(candidates, min(9, len(candidates)))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

COLORS = plt.cm.tab20.colors

for ax, img_info in zip(axes, sample_imgs):
    # Try to find the image file
    img_path = TACO_DIR / img_info['file_name']
    if not img_path.exists():
        img_path = TACO_DIR / 'images' / Path(img_info['file_name']).name
    if not img_path.exists():
        ax.set_visible(False)
        continue

    pil_img = Image.open(img_path).convert('RGB')
    iw, ih = pil_img.size
    ax.imshow(pil_img)

    labels = []
    for ann in img_to_anns.get(img_info['id'], []):
        x, y, w, h = ann['bbox']
        cat_name = cat_id_to_name.get(ann['category_id'], 'unknown')
        color = COLORS[ann['category_id'] % len(COLORS)]
        rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        ax.text(x, y - 4, cat_name, fontsize=6, color=color, fontweight='bold')
        labels.append(cat_name)

    ax.set_title(', '.join(set(labels))[:60], fontsize=8)
    ax.axis('off')

plt.suptitle('9 Random TACO Images with Annotations', fontsize=14)
plt.tight_layout()
plt.show()

## 3 — Dataset Health Checks

In [ ]:
# Images with no annotations
annotated_ids = set(a['image_id'] for a in annotations)
unannotated = [img for img in images if img['id'] not in annotated_ids]
print(f'Images with no annotations : {len(unannotated)} / {len(images)}')

# Very small bounding boxes (area < 1% of image area)
small_boxes = []
for ann in annotations:
    img = img_meta.get(ann['image_id'], {})
    iw = img.get('width', 1)
    ih = img.get('height', 1)
    x, y, w, h = ann['bbox']
    rel_area = (w * h) / (iw * ih)
    if rel_area < 0.01:
        small_boxes.append({
            'image_id': ann['image_id'],
            'class': cat_id_to_name.get(ann['category_id'], '?'),
            'rel_area': rel_area,
        })

print(f'Very small bboxes (<1% image area) : {len(small_boxes)}')
if small_boxes:
    tiny_classes = Counter(b['class'] for b in small_boxes)
    print('  Most common in tiny boxes:', tiny_classes.most_common(5))

# Class imbalance ratio
most_common_count = sorted_classes[0][1]
rarest_count      = sorted_classes[-1][1]
ratio = most_common_count / max(rarest_count, 1)
print(f'\nClass imbalance ratio (most / rarest) : {ratio:.1f}x')
print(f'  Most common : {sorted_classes[0][0]}  ({most_common_count} samples)')
print(f'  Rarest      : {sorted_classes[-1][0]}  ({rarest_count} samples)')

# Classes with fewer than 10 samples
rare = [(name, cnt) for name, cnt in sorted_classes if cnt < 10]
if rare:
    print(f'\nClasses with <10 annotations ({len(rare)}):')
    for name, cnt in rare:
        print(f'  {name:<40} {cnt}')
else:
    print('\nAll classes have ≥10 annotations ✓')

## 4 — Post-Training Analysis
Run this section **after** `python ml/train.py` has completed.

In [ ]:
import pandas as pd

results_csv = Path('data/models/taco_yolov8/results.csv')

if not results_csv.exists():
    print(f'results.csv not found at {results_csv}')
    print('Run python ml/train.py first, then re-run this cell.')
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    epochs = range(1, len(df) + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Train vs val box loss
    ax = axes[0]
    if 'train/box_loss' in df.columns:
        ax.plot(epochs, df['train/box_loss'], label='Train box loss', color='steelblue')
    if 'val/box_loss' in df.columns:
        ax.plot(epochs, df['val/box_loss'],   label='Val box loss',   color='tomato')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Box Loss')
    ax.legend()

    # Train vs val class loss
    ax = axes[1]
    if 'train/cls_loss' in df.columns:
        ax.plot(epochs, df['train/cls_loss'], label='Train cls loss', color='steelblue')
    if 'val/cls_loss' in df.columns:
        ax.plot(epochs, df['val/cls_loss'],   label='Val cls loss',   color='tomato')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Classification Loss')
    ax.legend()

    # mAP over epochs
    ax = axes[2]
    map_col = next((c for c in df.columns if 'mAP50' in c and '95' not in c), None)
    map95_col = next((c for c in df.columns if 'mAP50-95' in c or 'mAP_0.5:0.95' in c), None)
    if map_col:
        ax.plot(epochs, df[map_col],   label='mAP50',    color='green')
    if map95_col:
        ax.plot(epochs, df[map95_col], label='mAP50-95', color='darkolivegreen', linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('mAP over Epochs')
    ax.legend()

    plt.suptitle('Training Curves', fontsize=14)
    plt.tight_layout()
    plt.show()

    best_epoch = df[map_col].idxmax() + 1 if map_col else '?'
    best_map   = df[map_col].max()         if map_col else '?'
    print(f'Best mAP50 : {best_map:.4f} at epoch {best_epoch}')